<a href="https://colab.research.google.com/github/eishaiqbal767-cmyk/AI-TMBD-MOVIES-PROJECT/blob/main/Notebooks/TMBD_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mounting Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
BASE = '/content/drive/MyDrive/AI_Project'
print(os.path.exists(BASE))
print(os.listdir(BASE))

True
['notebooks', 'data', 'reports', 'src']


I mounted my Google Drive in Colab so the notebook can read and save files in my project folder.

In [6]:
import requests
from google.colab import userdata

TOKEN = userdata.get('TMDB_TOKEN')
BASE_URL = 'https://api.themoviedb.org/3'
HEADERS = {
    'Authorization': f'Bearer {TOKEN}',
    'accept': 'application/json'
}

params = {
    'primary_release_date.gte': '2016-01-01',
    'primary_release_date.lte': '2016-12-31',
    'sort_by': 'vote_count.desc',
    'page': 1
}
r = requests.get(f'{BASE_URL}/discover/movie', headers=HEADERS, params=params)
print('Status code:', r.status_code)
data = r.json()
print('Total results:', data['total_results'])
print('Total pages:', data['total_pages'])
print('Movies on this page:', len(data['results']))
print('First movie:', data['results'][0]['title'], '| id:', data['results'][0]['id'])

Status code: 200
Total results: 20001
Total pages: 1001
Movies on this page: 20
First movie: Deadpool | id: 293660


I made a first test request to the TMDB Discover endpoint to check that my API token works.

# TMDB Movie Data Analysis and Machine Learning

Programming for AI, semester project. Team: [Eisha Iqbal,25L-3003], [Ifrah noor, 25L-0025]. Instructor: Mahhek Tahir.

In this notebook I collect movie data from the TMDB API for the years 2016 to 2026, clean and explore it, and build regression models to predict a numerical target.

In [8]:
import os, re, json, time, hashlib, random
from datetime import datetime
import numpy as np
import pandas as pd

ROLL_NUMBERS = ['25L-3003', '25L-0025']

SEED = 25

random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/AI_Project'
RAW_DIR = f'{BASE}/data/raw'
PROC_DIR = f'{BASE}/data/processed'
JSON_DIR = f'{RAW_DIR}/json'
os.makedirs(JSON_DIR, exist_ok=True)

START_YEAR, END_YEAR = 2016, 2026
MOVIES_PER_YEAR = 100
SORT_BY = 'vote_count.desc'

print('Roll numbers:', ROLL_NUMBERS)
print('SEED:', SEED)
print('Raw data folder:', RAW_DIR)
print('Years:', START_YEAR, 'to', END_YEAR, '| target per year:', MOVIES_PER_YEAR)
print('Sorting:', SORT_BY)

Roll numbers: ['25L-3003', '25L-0025']
SEED: 25
Raw data folder: /content/drive/MyDrive/AI_Project/data/raw
Years: 2016 to 2026 | target per year: 100
Sorting: vote_count.desc


In [ ]:
COLLECTION_START = datetime.now()
request_count = 0
id_records = []
seen_ids = set()

def api_get(path, params=None):
    global request_count
    for attempt in range(5):
        r = requests.get(f'{BASE_URL}{path}', headers=HEADERS, params=params, timeout=30)
        request_count += 1
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
    raise RuntimeError('Too many failed attempts')

year_summary = []
for year in range(START_YEAR, END_YEAR + 1):
    collected, page, total_pages = 0, 1, None
    while collected < MOVIES_PER_YEAR:
        params = {
            'primary_release_date.gte': f'{year}-01-01',
            'primary_release_date.lte': f'{year}-12-31',
            'sort_by': SORT_BY,
            'vote_count.gte': 10,
            'include_adult': 'false',
            'language': 'en-US',
            'page': page
        }
        data = api_get('/discover/movie', params)
        total_pages = data['total_pages']
        for m in data['results']:
            if m['id'] in seen_ids:
                continue
            seen_ids.add(m['id'])
            id_records.append({'id': m['id'], 'title': m['title'],
                               'discover_year': year, 'discover_page': page})
            collected += 1
            if collected >= MOVIES_PER_YEAR:
                break
        if page >= total_pages:
            break
        page += 1
        time.sleep(0.05)
    year_summary.append({'year': year, 'collected': collected,'pages_used': page - 1
                         if collected >= MOVIES_PER_YEAR else page,
                         'api_total_pages': total_pages})

    print(year, '->', collected, 'movies')

ids_df = pd.DataFrame(id_records)
ids_df.to_csv(f'{RAW_DIR}/movie_ids.csv', index=False)
pd.DataFrame(year_summary).to_csv(f'{RAW_DIR}/discover_summary.csv', index=False)

print('\nUnique IDs:', ids_df['id'].nunique(), '| Requests so far:', request_count)
pd.DataFrame(year_summary)

2016 -> 100 movies
2017 -> 100 movies
2018 -> 100 movies
2019 -> 100 movies
2020 -> 100 movies
2021 -> 100 movies
2022 -> 100 movies
2023 -> 100 movies
2024 -> 100 movies
2025 -> 100 movies
2026 -> 100 movies

Unique IDs: 1100 | Requests so far: 55


,year,collected,pages_used,api_total_pages
0,2016,100,5,173
1,2017,100,5,185
2,2018,100,5,186
3,2019,100,5,184
4,2020,100,5,142
5,2021,100,5,148
6,2022,100,5,160
7,2023,100,5,153
8,2024,100,5,129
9,2025,100,5,102


The Discover step returned 1100 unique movie IDs, exactly 100 for each year from 2016 to 2026, with no duplicates. Each year needed 5 pages of 20 results, and the 11 years together used 55 requests.


In [ ]:
total = len(ids_df)
done_now, skipped = 0, 0

for i, movie_id in enumerate(ids_df['id'], start=1):
    path = f'{JSON_DIR}/{movie_id}.json'
    if os.path.exists(path):
        skipped += 1
        continue
    data = api_get(f'/movie/{movie_id}',
                   {'append_to_response': 'credits', 'language': 'en-US'})
    wrapper = {'collected_at': datetime.now().isoformat(timespec='seconds'),
               'data': data}
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(wrapper, f)
    done_now += 1
    if i % 100 == 0:
        print(f'{i}/{total} processed')
    time.sleep(0.05)

COLLECTION_END = datetime.now()
files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
print('\nDownloaded now:', done_now, '| skipped (already saved):', skipped)
print('JSON files on Drive:', len(files))
print('Total requests (discover + details):', request_count)
print('Start:', COLLECTION_START.isoformat(timespec='seconds'))
print('End:  ', COLLECTION_END.isoformat(timespec='seconds'))

with open(f'{RAW_DIR}/collection_log.json', 'w') as f:
    json.dump({'roll_numbers': ROLL_NUMBERS,
               'start': COLLECTION_START.isoformat(timespec='seconds'),
               'end': COLLECTION_END.isoformat(timespec='seconds'),
               'requests': request_count}, f)

100/1100 processed
200/1100 processed
300/1100 processed
400/1100 processed
500/1100 processed
600/1100 processed
700/1100 processed
800/1100 processed
900/1100 processed
1000/1100 processed
1100/1100 processed

Downloaded now: 1100 | skipped (already saved): 0
JSON files on Drive: 1100
Total requests (discover + details): 1155
Start: 2026-11-21T19:51:32
End:   2026-11-21T20:07:55


For each of the 1100 movie IDs I called the Movie Details endpoint and asked for the credits in the same request, so I got budget, revenue, runtime, genres, languages, production companies, cast and crew without extra calls.

In [10]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, re, json, time, hashlib, random, requests
from datetime import datetime
import numpy as np
import pandas as pd

ROLL_NUMBERS = ['25L-3003', '25L-0025']
SEED = 25
random.seed(SEED)
np.random.seed(SEED)

BASE = '/content/drive/MyDrive/AI_Project'
RAW_DIR = f'{BASE}/data/raw'
PROC_DIR = f'{BASE}/data/processed'
JSON_DIR = f'{RAW_DIR}/json'

START_YEAR, END_YEAR = 2016, 2026
MOVIES_PER_YEAR = 100
SORT_BY = 'vote_count.desc'

TOKEN = userdata.get('TMDB_TOKEN')
BASE_URL = 'https://api.themoviedb.org/3'
HEADERS = {'Authorization': f'Bearer {TOKEN}', 'accept': 'application/json'}

ids_df = pd.read_csv(f'{RAW_DIR}/movie_ids.csv')
print('IDs reloaded:', len(ids_df))
print('JSON files on Drive:', len([f for f in os.listdir(JSON_DIR) if f.endswith('.json')]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IDs reloaded: 1100
JSON files on Drive: 1100


The Colab session restarted, so the variables in memory were lost. I remounted Google Drive, restored my settings, and reloaded the list of movie IDs from the file I had saved earlier.

In [12]:
def names(items, key='name'):
    if not items:
        return None
    return '|'.join(str(x[key]) for x in items if x.get(key))

rows = []
for movie_id, disc_year in zip(ids_df['id'], ids_df['discover_year']):
    with open(f'{JSON_DIR}/{movie_id}.json', encoding='utf-8') as f:
        w = json.load(f)
    d, credits = w['data'], w['data'].get('credits', {})
    crew, cast = credits.get('crew', []), credits.get('cast', [])
    directors = [c['name'] for c in crew if c.get('job') == 'Director']
    coll = d.get('belongs_to_collection')
    rows.append({
        'id': d['id'],
        'title': d.get('title'),
        'original_title': d.get('original_title'),
        'release_date': d.get('release_date'),
        'discover_year': disc_year,
        'status': d.get('status'),
        'original_language': d.get('original_language'),
        'runtime': d.get('runtime'),
        'budget': d.get('budget'),
        'revenue': d.get('revenue'),
        'popularity': d.get('popularity'),
        'vote_average': d.get('vote_average'),
        'vote_count': d.get('vote_count'),
        'adult': d.get('adult'),
        'video': d.get('video'),
        'genres': names(d.get('genres')),
        'production_companies': names(d.get('production_companies')),
        'production_countries': names(d.get('production_countries')),
        'spoken_languages': names(d.get('spoken_languages'), 'english_name'),
        'collection_name': coll['name'] if coll else None,
        'director': directors[0] if directors else None,
        'top_cast': '|'.join(c['name'] for c in cast[:3]) or None,
        'cast_size': len(cast),
        'crew_size': len(crew),
        'tagline': d.get('tagline') or None,
        'overview': d.get('overview') or None,
        'imdb_id': d.get('imdb_id'),
        'collected_at': w['collected_at'],
    })

raw_df = pd.DataFrame(rows)
RAW_CSV = f'{RAW_DIR}/tmdb_raw.csv'
raw_df.to_csv(RAW_CSV, index=False, encoding='utf-8')

print('Rows:', raw_df.shape[0], '| Columns:', raw_df.shape[1])
print('Duplicate IDs:', raw_df['id'].duplicated().sum())
raw_df.head()

Rows: 1100 | Columns: 28
Duplicate IDs: 0


,id,title,original_title,release_date,discover_year,status,original_language,runtime,budget,revenue,...,spoken_languages,collection_name,director,top_cast,cast_size,crew_size,tagline,overview,imdb_id,collected_at
0,293660,Deadpool,Deadpool,2016-02-09,2016,Released,en,108,58000000,782837347,...,English,Deadpool Collection,Tim Miller,Ryan Reynolds|Morena Baccarin|Ed Skrein,52,489,Feel the love.,The origin story of former Special Forces oper...,tt1431045,2026-11-21T19:53:17
1,271110,Captain America: Civil War,Captain America: Civil War,2016-04-27,2016,Released,en,147,250000000,1155046416,...,Romanian|English|German|Russian,Captain America Collection,Joe Russo,Chris Evans|Robert Downey Jr.|Scarlett Johansson,104,296,United we stand. Divided we fall.,"Following the events of Age of Ultron, the col...",tt3498820,2026-11-21T19:53:17
2,284052,Doctor Strange,Doctor Strange,2016-10-25,2016,Released,en,115,180000000,676343174,...,English,Doctor Strange Collection,Scott Derrickson,Benedict Cumberbatch|Chiwetel Ejiofor|Rachel M...,44,253,The impossibilities are endless.,"After his career is destroyed, a brilliant but...",tt1211837,2026-11-21T19:53:18
3,297761,Suicide Squad,Suicide Squad,2016-08-03,2016,Released,en,122,175000000,749200054,...,English|Japanese|Spanish,Suicide Squad Collection,David Ayer,Will Smith|Jared Leto|Margot Robbie,106,194,Worst. Heroes. Ever.,"From DC Comics comes the Suicide Squad, an ant...",tt1386697,2026-11-21T19:53:19
4,329865,Arrival,Arrival,2016-11-10,2016,Released,en,116,47000000,203388186,...,Mandarin|English|Russian,None,Denis Villeneuve,Amy Adams|Jeremy Renner|Forest Whitaker,72,487,Why are they here?,Taking place after alien crafts land around th...,tt2543164,2026-11-21T19:53:20


I read the 1100 saved JSON files and combined them into a single table with one movie per row, then saved it as tmdb_raw.csv on Google Drive. I chose attributes from several groups: numbers such as budget, revenue, runtime and vote count, categories such as language, status and genre, dates such as the release date, and true or false values such as adult and video. Fields that hold several values, for example genres and production companies, were joined into one text value separated by a vertical bar so I can split them later during cleaning. I also took the director and the three top billed actors from the credits, and I kept a collected_at timestamp in every row. I did not change any values in this step, so zero budgets and zero revenues are still there and I will report them in the data cleaning section. The output shows 1100 rows and no repeated IDs.

In [15]:
log = {
    'roll_numbers': ROLL_NUMBERS,
    'start': '2026-11-21T19:51:32',
    'end': '2026-11-21T20:58:43',
    'requests': 1210,
    'note': ('First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). '
             'The discover step was rerun at 20:58 (55 requests) to fix a summary column; '
             'the 1100 saved detail files were reused, not downloaded again.')
}
with open(f'{RAW_DIR}/collection_log.json', 'w') as f:
    json.dump(log, f)
print(log)

{'roll_numbers': ['25L-3003', '25L-0025'], 'start': '2026-11-21T19:51:32', 'end': '2026-11-21T20:58:43', 'requests': 1210, 'note': 'First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). The discover step was rerun at 20:58 (55 requests) to fix a summary column; the 1100 saved detail files were reused, not downloaded again.'}


I corrected the saved collection log file so that it matches what really happened during the collection. When I reran the Discover step to fix a page count in my summary table, the log file was overwritten with the new start time and a request count of only 55. I restored the real values: the collection started at 19:51 and the last rerun ended at 20:58, and the total was 1210 requests, which is 1155 from the first full run plus 55 from the Discover rerun.  The output shows the corrected values. The dataset itself was not changed, because the 1100 detail files had already been saved and were reused.

In [16]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

with open(f'{RAW_DIR}/collection_log.json') as f:
    log = json.load(f)

fingerprint = sha256_of(RAW_CSV)

print('=== DATA COLLECTION LOG ===')
print('Team roll numbers      :', ', '.join(log['roll_numbers']))
print('Source                 : TMDB API', BASE_URL)
print('Endpoints used         : /discover/movie, /movie/{id}?append_to_response=credits')
print('Collection start       :', log['start'])
print('Collection end         :', log['end'])
print('Number of requests     :', log['requests'], '(55 + 1100 first run, 55 discover rerun)')
print('Note                   :', log['note'])
print('Rows collected         :', len(raw_df))
print('collected_at column    : present | earliest', raw_df['collected_at'].min(),
      '| latest', raw_df['collected_at'].max())
print('Raw file               :', RAW_CSV)
print('SHA-256 fingerprint    :', fingerprint)

with open(f'{RAW_DIR}/data_collection_log.txt', 'w') as f:
    f.write(f"Roll numbers: {log['roll_numbers']}\nStart: {log['start']}\nEnd: {log['end']}\n"
            f"Requests: {log['requests']}\nNote: {log['note']}\nRows: {len(raw_df)}\nSHA-256: {fingerprint}\n")

=== DATA COLLECTION LOG ===
Team roll numbers      : 25L-3003, 25L-0025
Source                 : TMDB API https://api.themoviedb.org/3
Endpoints used         : /discover/movie, /movie/{id}?append_to_response=credits
Collection start       : 2026-11-21T19:51:32
Collection end         : 2026-11-21T20:58:43
Number of requests     : 1210 (55 + 1100 first run, 55 discover rerun)
Note                   : First full collection 19:51-20:08: 55 discover + 1100 detail requests (1155). The discover step was rerun at 20:58 (55 requests) to fix a summary column; the 1100 saved detail files were reused, not downloaded again.
Rows collected         : 1100
collected_at column    : present | earliest 2026-11-21T19:53:17 | latest 2026-11-21T20:07:55
Raw file               : /content/drive/MyDrive/AI_Project/data/raw/tmdb_raw.csv
SHA-256 fingerprint    : 45903e104914bc20dc3352cf6914673c628396a9ecb1d7fd110cd92a167b5447


This cell is my data collection log. It records my team's roll numbers, the source and endpoints, the start and end time, the number of requests, the number of rows, the collected_at column and the SHA-256 fingerprint of the raw CSV file. The first full collection ran from 19:51 to about 20:08 and used 1155 requests. The Discover step was rerun once with 55 more requests, and the 1100 detail files I had already saved were reused, so the log records 1210 requests in total. The collected_at values in the rows still show the original collection time. The fingerprint is calculated from the raw CSV file itself, so it works as proof that the file I submit is the one my own collection produced. From now on I will not overwrite tmdb_raw.csv, and all cleaning will be done on copies.